In [1]:
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, f1_score
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import SMOTE
import os
import time

In [ ]:
# Constants
NUM_CLASSES = 4
RANDOM_STATE = 1082
MODEL_NAME = "XGBoost (SMOTE)"


In [ ]:
import sys as _sys
from pathlib import Path as _Path

_root = _Path.cwd().resolve()
while not (_root / "eval_metrics.py").is_file() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in _sys.path:
    _sys.path.insert(0, str(_root))
from gps_ids import protocol


In [4]:
print("First 5 rows of the dataset:")
print(df.head())
print("\nDataset information:")
print(df.info())

First 5 rows of the dataset:
   PRN           DO             PD         RX            TOW            CP  \
0    6  1160.671408 -238359.543282  491568.00  491568.000795 -24660.263293   
1    6  1157.827808 -238363.921192  491568.02  491568.020795 -24683.468520   
2    6  1161.135932 -238368.638615  491568.04  491568.040795 -24706.687357   
3    6  1161.156617 -238373.391775  491568.06  491568.060795 -24729.920039   
4    6  1160.690893 -238378.127066  491568.08  491568.080795 -24753.153999   

          EC         LC         PC        PIP          PQP        TCD  \
0  163521.78  170008.81  179294.97  178420.33 -17688.17800  1158.4806   
1  158277.05  160253.09  182106.11  182106.05    158.33333  1160.7948   
2  184442.36  193125.89  208228.02 -205198.30  35391.59000  1157.6909   
3  159812.45  169960.55  187550.72  187543.00  -1701.94400  1161.1343   
4  183557.66  191912.58  208403.50  208330.05  -5532.89990  1161.1759   

         CN0  Output  
0  49.412529       0  
1  49.452686     

In [ ]:
EXP = protocol.smote_heldout("2d")
X_train_resampled = EXP.X_train
y_train_resampled = EXP.y_train
X_test_scaled = EXP.X_test
y_test = EXP.y_test


In [6]:
# Train the XGBoost model
print("\nTraining the XGBoost model...")

model = XGBClassifier(
    objective='multi:softprob',
    num_class=NUM_CLASSES,
    eval_metric='mae',
    random_state=RANDOM_STATE,
)
train_start_time = time.time()
model.fit(X_train_resampled, y_train_resampled)
train_time = time.time() - train_start_time

print("XGBoost model trained successfully!")


Training the XGBoost model...
XGBoost model trained successfully!


In [9]:
# Evaluate the model

print("\nEvaluating the model on the test set...")
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
print(f"Model Accuracy on Test Set: {accuracy * 100:.2f}%")
print(f"F1 Score: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))


Evaluating the model on the test set...
Model Accuracy on Test Set: 93.54%
F1 Score: 0.9390

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.94      0.97    318260
           1       0.76      0.91      0.83     29166
           2       0.68      0.89      0.77     35386
           3       0.92      0.99      0.95     25612

    accuracy                           0.94    408424
   macro avg       0.84      0.93      0.88    408424
weighted avg       0.95      0.94      0.94    408424



In [ ]:
from eval_metrics import evaluate, summary, time_inference
from gps_ids.results import append_validated

_proba = model.predict_proba(X_test_scaled) if hasattr(model, "predict_proba") else None
_timing = time_inference(model.predict, X_test_scaled)

row = evaluate(
    y_test,
    y_pred,
    model_name=MODEL_NAME,
    family="SMOTE",
    y_proba=_proba,
    train_time_sec=train_time,
    num_epochs="N/A",
    parameters=model.get_params(),
    timing=_timing,
    **EXP.eval_kwargs(),
)
append_validated(row, EXP.results_csv)
print(summary(row))
print(f"Results appended to {EXP.results_csv}")
